Extract Hitorical Financial Data

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.stattools import adfuller

# Define the tickers and the date range
tickers = ['TSLA', 'BND', 'SPY']
start_date = '2015-07-01'
end_date = '2025-07-31'

# Fetch the data
data = yf.download(tickers, start=start_date, end=end_date)

# Separate the data for each ticker
tsla_data = data['Adj Close']['TSLA'].to_frame().rename(columns={'TSLA': 'Adj Close'})
tsla_data[['Open', 'High', 'Low', 'Close', 'Volume']] = data[['Open', 'High', 'Low', 'Close', 'Volume']]['TSLA']

bnd_data = data['Adj Close']['BND'].to_frame().rename(columns={'BND': 'Adj Close'})
bnd_data[['Open', 'High', 'Low', 'Close', 'Volume']] = data[['Open', 'High', 'Low', 'Close', 'Volume']]['BND']

spy_data = data['Adj Close']['SPY'].to_frame().rename(columns={'SPY': 'Adj Close'})
spy_data[['Open', 'High', 'Low', 'Close', 'Volume']] = data[['Open', 'High', 'Low', 'Close', 'Volume']]['SPY']

print("TSLA Data Head:")
print(tsla_data.head())

Data Cleaning and Understanding

In [ ]:
# Check for missing values
print("Missing values in TSLA data:\n", tsla_data.isnull().sum())

# Display basic statistics
print("\nTSLA Data Description:\n", tsla_data.describe())

EDA

In [ ]:
# Plot the Adjusted Close price
plt.figure(figsize=(14, 7))
plt.plot(tsla_data['Adj Close'], label='TSLA Adj Close')
plt.title('TSLA Adjusted Close Price Over Time')
plt.xlabel('Date')
plt.ylabel('Adjusted Close Price (USD)')
plt.legend()
plt.grid(True)
plt.savefig('../results/plots/tsla_adj_close.png')
plt.show()

# Calculate and plot daily percentage change
tsla_data['Daily Return'] = tsla_data['Adj Close'].pct_change()
plt.figure(figsize=(14, 7))
plt.plot(tsla_data['Daily Return'], label='TSLA Daily Return', color='orange')
plt.title('TSLA Daily Returns')
plt.xlabel('Date')
plt.ylabel('Percentage Change')
plt.legend()
plt.grid(True)
plt.savefig('../results/plots/tsla_daily_return.png')
plt.show()

Analyze Volatility

# Calculate 30-day rolling mean and standard deviation
tsla_data['Rolling Mean'] = tsla_data['Adj Close'].rolling(window=30).mean()
tsla_data['Rolling Std'] = tsla_data['Adj Close'].rolling(window=30).std()

# Plot rolling statistics
plt.figure(figsize=(14, 7))
plt.plot(tsla_data['Adj Close'], label='TSLA Adj Close')
plt.plot(tsla_data['Rolling Mean'], label='30-Day Rolling Mean')
plt.plot(tsla_data['Rolling Std'], label='30-Day Rolling Std')
plt.title('TSLA Rolling Mean and Standard Deviation')
plt.xlabel('Date')
plt.ylabel('Price (USD)')
plt.legend()
plt.grid(True)
plt.savefig('../results/plots/tsla_rolling_stats.png')
plt.show()

Augemented-Dickey Fuller Test

# Function to perform the ADF test
def adf_test(series, name=''):
    result = adfuller(series.dropna())
    print(f'ADF Test for {name}:')
    print(f'ADF Statistic: {result[0]}')
    print(f'p-value: {result[1]}')
    print('Critical Values:')
    for key, value in result[4].items():
        print(f'\t{key}: {value}')

# Perform ADF test on Adjusted Close and Daily Returns
adf_test(tsla_data['Adj Close'], 'TSLA Adjusted Close')
print("\n")
adf_test(tsla_data['Daily Return'], 'TSLA Daily Return')

Risk Metrics

# Calculate VaR
confidence_level = 0.95
VaR = tsla_data['Daily Return'].quantile(1 - confidence_level)
print(f"95% Value at Risk (VaR): {VaR:.4f}")

# Calculate Sharpe Ratio
risk_free_rate = 0.0 # Assuming a risk-free rate of 0 for simplicity
sharpe_ratio = (tsla_data['Daily Return'].mean() * 252) / (tsla_data['Daily Return'].std() * np.sqrt(252))
print(f"Sharpe Ratio: {sharpe_ratio:.4f}")